In [ ]:
import tables_io
import utility_functions
import numpy as np
import matplotlib.pyplot as plt
import admixture_functions

#### Download and uncompress the data

In [ ]:
_ = utility_functions.setup_data()

#### Change this to match the correct location

In [ ]:
pz_dir = '.'

## Exercise 1: compare training with representative and non-representative data

The goal of this exercise is to understand the effect of using non-representative data for traning.

At some level this is inevitable when making photometric redshifts: if we could easily get spectra at the same depths to which we can get imaging data and photometry, then we would not need to make photometric redshifts.

To do this we will use files that TQ has prepared extracted from the PZ data challenge preperation.  Specifically files with the 

In [ ]:
train_rep_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/train_representative_lsstY10_ugrizy_100k.pq')
train_non_rep_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/train_nonrepresentative_specz_lsstY10_ugrizy_100k.pq')
test_data = tables_io.read(f'{pz_dir}/data/task1_representativeness/test_lsstY10_ugrizy_100k.pq')

In [ ]:
train_rep_data_targets, train_rep_data_features = utility_functions.prepare_data_total_mag_and_colors(train_rep_data, 'mag_{band}_lsst', 'ugrizy')
train_non_rep_data_targets, train_non_rep_data_features = utility_functions.prepare_data_total_mag_and_colors(train_non_rep_data, 'mag_{band}_lsst', 'ugrizy')
test_targets, test_features = utility_functions.prepare_data_total_mag_and_colors(test_data, 'mag_{band}_lsst', 'ugrizy')

In [ ]:
redshift_bins = np.linspace(0., 2.2, 111)
_ = plt.hist(train_rep_data_targets, bins=redshift_bins, alpha=0.2, label='Train (rep)')
_ = plt.hist(train_non_rep_data_targets, bins=redshift_bins, alpha=0.2, label='Train (non-rep)')
_ = plt.hist(test_targets, bins=redshift_bins, alpha=0.2, label='Test')
_ = plt.legend()
_ = plt.xlabel('z')
_ = plt.ylabel('Objects / 0.02')

## Exercise 2: explore effects of adding near IR bands to optical bands when making photometric redshifts

The goal of this exercise is to understand the effect of using near IR bands (e.g., from Roman) as well as optical bands.

To do this we will use more files that TQ has prepared extracted from the PZ data challenge preperation.

In [ ]:
train_optical_data = tables_io.read(f'{pz_dir}/data/task2_nir/train_lsstY10_ugrizy_100k.pq')
train_opt_nir_data = tables_io.read(f'{pz_dir}/data/task2_nir/train_lsstY10_ugrizy_romanYJH_100k.pq')
test_optical_data = tables_io.read(f'{pz_dir}/data/task2_nir/test_lsstY10_ugrizy_100k.pq')
test_opt_nir_data = tables_io.read(f'{pz_dir}/data/task2_nir/test_lsstY10_ugrizy_romanYJH_100k.pq')

In [ ]:
train_opt_nir_data

In [ ]:
optical_bands = ['u', 'g', 'r', 'i', 'z', 'y']
optical_and_nir_bamds = ['u_lsst', 'g_lsst', 'r_lsst', 'i_lsst', 'z_lsst', 'y_lsst', 'Y106_roman', 'J129_roman', 'H158_roman']

train_optical_targets, train_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    train_optical_data, 'mag_{band}_lsst', optical_bands,
)
test_optical_targets, test_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    test_optical_data, 'mag_{band}_lsst', optical_bands,
)


train_opt_nir_targets, train_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    train_opt_nir_data, 'mag_{band}', optical_and_nir_bamds,
)
test_opt_nir_targets, test_optical_features = utility_functions.prepare_data_total_mag_and_colors(
    test_opt_nir_data, 'mag_{band}', optical_and_nir_bamds,
)

## Exercise 3: add morpology information to training features

In [ ]:
train = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_train.hdf5')
test = tables_io.read(f'{pz_dir}/data/lsst_cat_matched_nonan_test.hdf5')


In [ ]:
def make_mask(t):
    mask = (t['refExtendedness'] == 1) *\
           (t['i_blendedness'] < 0.01) *\
           (t['g_psfFlux']/ t['g_psfFluxErr'] > 10) *\
           (t['i_psfFlux'] / t['i_psfFluxErr'] > 10) *\
           (t['r_psfFlux'] / t['r_psfFluxErr'] > 10) *\
           (t['z_psfFlux'] / t['z_psfFluxErr'] > 10)
    return mask

In [ ]:
mask_test = make_mask(test)
mask_train = make_mask(train)

In [ ]:
def get_features(t):
    """ This will include the sersic fit index,    
    the trace in g and z bands, and the trace of the sersic fits
    in addition to the magntidues
    """
    sersic_x = t['sersic_reff_x']  
    sersic_y = t['sersic_reff_y']
    sersic_trace = np.nan_to_num(sersic_x*sersic_x + sersic_y*sersic_y, nan=0)
    sersic_index = np.nan_to_num(t['sersic_index'], 0)

    g_trace = np.nan_to_num(t['g_ixx'] + t['g_iyy'], nan=0)
    z_trace = np.nan_to_num(t['z_ixx'] + t['z_iyy'], nan=0)

    feature_list = [sersic_index, g_trace, z_trace, sersic_trace]
    feature_list += [np.nan_to_num(t[f'{band}_psfMag'], nan=30) for band in 'ugrizy']
    
    return np.vstack(feature_list).T


In [ ]:
features_test = get_features(test)[mask_test]
features_train = get_features(train)[mask_train]

targets_test = test['redshift'][mask_test]
targets_train = train['redshift'][mask_train]

## Exercise 4: explore effects of random photometric errors and/or flux contamination from blending

In [ ]:
train = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_train.hdf5")
test = tables_io.read(f"{pz_dir}/data/dp1_matched_v4_test.hdf5")

In [ ]:
train_features = utility_functions.get_band_values(train, '{band}_gaap1p0Mag', 'ugrizy')
train_targets = train['redshift']
test_features = utility_functions.get_band_values(test, '{band}_gaap1p0Mag', 'ugrizy')
test_targets = test['redshift']

library = utility_functions.get_band_values(test, '{band}_gaap1p0Mag', 'ugrizy')

In [ ]:
noise_grid = np.logspace(-4, 0, 17)
def make_noised_mags(features):
    the_dict = {}
    inputs = np.hstack([features])

    outputs = []
    for noise in noise_grid:
        noisy_mags = admixture_functions.gaussian_noise(features, noise_levels=noise*np.ones(6)).T
        outputs.append(noisy_mags)
    return outputs

In [ ]:
admix_grid = np.logspace(-4, 0, 17)
def make_admix_mags(features):
    the_dict = {}
    inputs = np.hstack([features])
    outputs = []

    for admix in admix_grid:
        mixed_mags = admixture_functions.make_admixture(train_features, library, admixture=admix)
        outputs.append(mixed_mags)
    return outputs

In [ ]:
noised_mags = make_noised_mags(train_features)
admix_mags = make_admix_mags(train_features)

In [ ]:
noise_delta = np.zeros((17, 6))
noise_rms = np.zeros((17, 6))
admix_delta = np.zeros((17, 6))
admix_rms = np.zeros((17, 6))

for i in range(17):
    deltas_noise = train_features - noised_mags[i]
    deltas_admix = train_features - admix_mags[i]
    noise_delta[i] = deltas_noise.mean(axis=0)
    noise_rms[i] = deltas_noise.std(axis=0)
    admix_delta[i] = deltas_admix.mean(axis=0)
    admix_rms[i] = deltas_admix.std(axis=0)


In [ ]:
_ = plt.scatter(noise_grid, noise_rms.mean(axis=1), label='scatter' )
_ = plt.scatter(admix_grid, admix_rms.mean(axis=1))
_ = plt.scatter(admix_grid, admix_delta.mean(axis=1))
_ = plt.xscale('log')

## Exercise 5: comparing results from different algorithms

## Excercise 6: going from per-object p(z) to ensemble n(z)